# Notebook 018 — The Crypto Perpetual Funding Basis Trade (Gate FA)

Long spot, short the perpetual, same symbol, delta-neutral by construction, collecting the 8-hourly
funding payment as the return. The one candidate from notebook 009's own Phase 3 shortlist that had
never been run, parked on a data blocker ("needs a spot price series this repo's cached data was not
verified to include") that this notebook resolves. See `src/results/018_funding_basis_trade.md` for
the full write-up; this notebook presents the same results by loading the committed JSON each phase
produced, not by re-running the backtests inline.

In [1]:
import json
import sys

sys.path.insert(0, "..")
sys.path.insert(0, "tmp")

TMP = "tmp"


def load(name):
    with open(f"{TMP}/{name}") as f:
        return json.load(f)


prereg = load("phase_0_18_preregistration.json")
print("Gates pre-registered:", [g["id"] for g in prereg["gates"]])
print("n_trials baseline:", prereg["n_trials"]["baseline"])
print()
print("Frozen constants:")
for k in [
    "ROUND_TURN_BP",
    "THETA_IN",
    "THETA_OUT",
    "CARRY_EWMA_HALF_LIFE_PERIODS_N",
    "TARGET_HOLD_PERIODS_H",
    "MAX_POSITIONS_N_MAX",
    "LIQUIDITY_FLOOR_USD_PER_DAY",
]:
    print(f"  {k}: {prereg['constants'][k]}")

Gates pre-registered: ['FA-1', 'FA-2', 'FA-3', 'FA-4', 'FUND']
n_trials baseline: 18

Frozen constants:
  ROUND_TURN_BP: 34.0
  THETA_IN: 7.55555555555556e-05
  THETA_OUT: 3.77777777777778e-05
  CARRY_EWMA_HALF_LIFE_PERIODS_N: 21
  TARGET_HOLD_PERIODS_H: 45
  MAX_POSITIONS_N_MAX: 10
  LIQUIDITY_FLOOR_USD_PER_DAY: 5000000


## Phase 0 — The two objections this notebook must answer before it runs

Answered in prose in the pre-registration, before any result existed.

In [2]:
print("2.1: 007 Phase C already tested funding carry?")
print(prereg["objection_2_1"]["answer"][:600] + "...")
print()
print("2.2: isn't this just crypto beta?")
print(prereg["objection_2_2"]["answer"])

2.1: 007 Phase C already tested funding carry?
No -- 007 Phase C tested a structurally different trade: a cross-sectional, dollar-neutral book in perpetuals ONLY (long low-funding names, short high-funding names), which is a bet that funding predicts price DIRECTION. It was net Sharpe-negative in all 24 configurations, with realized turnover ~674-681/yr (about 40% higher than the price-based signal it was compared to) because a rank-based book churns whenever cross-sectional funding rates cluster. Gate FA is a per-symbol, spot+perp, delta-neutral position that collects the funding PAYMENT itself; there is no cross-section to rank against a...

2.2: isn't this just crypto beta?
If it is, Gate FA-4 catches it and the notebook fails on validity grounds regardless of Sharpe. A basis book with |beta| to the equal-weight crypto basket or to BTC >= 0.10 (on net returns) is not the delta-neutral trade this notebook claims to test, and the most likely cause of a beta leak is an implementation b

## Phase 1 — Data: the 009 blocker, resolved

Spot, futures/um, and `premiumIndexKlines` 8h monthly archives all verified live. 126 of 128
universe-seed symbols have a usable spot leg; two do not (`1000SHIBUSDT`, `1000XECUSDT`) — a capacity
finding, not an error. A live transient DNS failure mid-fetch (116/256 symbol-windows) recovered
completely via the idempotent cache on a resumed run.

In [3]:
phase3 = load("phase_3_18_results.json")
manifest_summary = phase3["universe_manifest_summary"]
print(
    f"Universe: {manifest_summary['n_ok']} ok / {manifest_summary['total_seed_symbols']} seed symbols"
)
no_spot = [s for s, v in manifest_summary["manifest"].items() if v == "no_spot"]
print("no_spot:", no_spot)

Universe: 126 ok / 128 seed symbols
no_spot: ['1000SHIBUSDT', '1000XECUSDT']


## Phase 2 — Library (`basis_lib18.py`)

`carry_estimate` (causal EWMA, half-life not span), `paired_log_return` (explicit two-leg identity),
`qualifies` (per-symbol absolute-threshold hysteresis — not `alpha_lib7.hysteresis_weights`, which is
cross-sectional/rank-based and would reimport 007 Phase C's own failure mode), and
`apply_two_leg_costs` (explicit spot+perp fee accounting). Six required tests plus a cross-check
against `research.add_portfolio_costs`, all green — see `tests/test_basis_lib18.py`.

## Phase 3 — Mechanism probe (no cost model, no Sharpe, no strategy verdict except FA-1)

Two real bugs found and fixed here (see `src/results/018_funding_basis_trade.md`'s "Bugs found"
section for the full account): a units error in the break-even-periods constant (1133 instead of 34),
and a small number of price-feed-artifact bars (`DGBUSDT`, `LUNAUSDT`) distorting two pooled
statistics, fixed by excluding `|basis| > 20%` bars from descriptive checks with the exclusion count
reported.

In [4]:
print(
    "Gate FA-1:",
    phase3["gate_fa1"]["fires"],
    f"(mean={phase3['gate_fa1']['pooled_mean_gross_paired_return']:.2e}, "
    f"NW t={phase3['gate_fa1']['newey_west_t']:.2f})",
)
print()
decomp = phase3["check_1_decomposition"]
print(
    f"Funding term: mean={decomp['funding_term_pooled_mean']:.2e}, t={decomp['funding_term_newey_west_t']:.2f}"
)
print(
    f"Basis-change term: mean={decomp['basis_change_term_pooled_mean']:.2e}, t={decomp['basis_change_term_newey_west_t']:.2f}"
)
print(
    f"Funding dominates by significance: {decomp['funding_dominates_by_significance']}"
)
print(
    f"Excluded {decomp['n_obs_excluded_implausible_basis']} implausible-basis obs "
    f"({len(decomp['symbols_with_excluded_bars'])} symbols touched)"
)
print()
crosscheck = phase3["check_3_premium_crosscheck"]
print(
    f"Premium crosscheck: median per-symbol corr={crosscheck['per_symbol_correlation_median']:.3f}, "
    f"materially_disagree={crosscheck['materially_disagree']}"
)
print()
persist = phase3["check_4_persistence"]
print(
    f"Persistence: {persist['frac_runs_clearing_breakeven']:.1%} of carry-above-theta_in runs "
    f"clear the {persist['breakeven_periods_used']}-period breakeven "
    f"(median run length {persist['median_run_length_periods']} periods)"
)
print()
print("Pooled funding by year:")
for row in phase3["check_2_pooled_funding_by_year"]["by_year"]:
    print(
        f"  {row['year']}: mean={row['newey_west_mean']:.2e}, t={row['newey_west_t']:.2f}, n={row['n_obs']}"
    )

Gate FA-1: True (mean=4.30e-05, NW t=3.27)

Funding term: mean=4.75e-05, t=20.50
Basis-change term: mean=1.05e-06, t=0.33
Funding dominates by significance: True
Excluded 5066 implausible-basis obs (14 symbols touched)

Premium crosscheck: median per-symbol corr=0.744, materially_disagree=False

Persistence: 44.0% of carry-above-theta_in runs clear the 34-period breakeven (median run length 25.0 periods)

Pooled funding by year:
  2021: mean=1.41e-04, t=25.82, n=58776
  2022: mean=-4.22e-05, t=-11.74, n=133751
  2023: mean=3.60e-05, t=6.13, n=136082
  2024: mean=1.10e-04, t=45.64, n=132806


## Phase 4 — The backtest (timed / always-on / cash, 4 origin offsets)

All four origin offsets agree to 3+ decimals — vacuous for this fixed-parameter, non-refit design,
same pattern 012 and Design A found.

In [5]:
phase4 = load("phase_4_18_results.json")
o0 = phase4["by_offset"]["0"]
print(
    f"{'book':<12}{'gross Sharpe':>14}{'net Sharpe':>14}{'net MDD':>10}{'turnover/yr':>14}"
)
for label, key in [("timed", "timed"), ("always_on", "always_on")]:
    m = o0[key]
    print(
        f"{label:<12}{m['sharpe']:>14.3f}{m['sharpe_net']:>14.3f}{m['max_drawdown_net']:>10.1%}{m['annualized_turnover']:>14.1f}"
    )
print(f"{'cash':<12}{0.0:>14.3f}{0.0:>14.3f}{0.0:>10.1%}{0.0:>14.1f}")
print()
print("Bootstrap CIs (offset 0):")
print(f"  timed net return 95% CI: {o0['timed_net_bootstrap_ci_95']}")
print(
    f"  (timed - always_on) net return 95% CI: {o0['timed_minus_always_on_bootstrap_ci_95']}"
)
print(
    f"  beta to crypto basket: {o0['beta_to_crypto_basket']:.4f}, beta to BTC: {o0['beta_to_btc']:.4f}"
)
print()
print(
    "DSR:",
    phase4["dsr"]["deflated_sharpe_prob"],
    f"(n_trials={phase4['n_trials_used']}, skew={phase4['dsr']['sample_skew']:.1f}, "
    f"kurtosis={phase4['dsr']['sample_kurtosis']:.1f})",
)
print()
print("Gates:", json.dumps(phase4["gates"], indent=2))
print()
print("Holdout access:", phase4["holdout_access"])

book          gross Sharpe    net Sharpe   net MDD   turnover/yr
timed                2.661         0.577     -8.6%          56.9
always_on            0.209        -0.415    -25.0%          15.2
cash                 0.000         0.000      0.0%           0.0

Bootstrap CIs (offset 0):
  timed net return 95% CI: [-1.3111946383579639e-05, 7.023274365216777e-05]
  (timed - always_on) net return 95% CI: [-1.0346173931084859e-05, 0.00010458522987067255]
  beta to crypto basket: 0.0005, beta to BTC: 0.0016

DSR: 0.18590973717553716 (n_trials=18, skew=-11.5, kurtosis=816.9)

Gates: {
  "FA-2": {
    "sharpe_leg_fires": true,
    "bootstrap_ci_leg_fires": false,
    "dsr_leg_fires": false,
    "fires": false,
    "fires_except_dsr_leg": false
  },
  "FA-3": {
    "fires": false
  },
  "FA-4": {
    "fires": true
  },
  "FUND": {
    "fires": false,
    "max_drawdowns_net_by_offset": {
      "0": -0.08613521755565864,
      "1": -0.08613521755565864,
      "2": -0.08613521755565864,
      "3":

In [6]:
c = phase4["concentration_diagnostic"]
print(
    f"Median symbols held: {c['n_symbols_held_median']}, at cap {c['frac_bars_at_cap']:.1%} of bars, "
    f"single-symbol {c['frac_bars_single_symbol']:.1%} of bars"
)
print()
print("Worst 5 bars (net return, symbols held):")
for w in c["worst_5_bars"]:
    print(
        f"  {w['datetime']}: {w['trade_log_return_net']:.4f}  {w['symbols_held_at_decision']}"
    )

Median symbols held: 10.0, at cap 53.8% of bars, single-symbol 5.4% of bars

Worst 5 bars (net return, symbols held):
  2022-06-25 00:00:00: -0.0568  ['ICPUSDT']
  2024-09-05 00:00:00: -0.0200  ['MATICUSDT']
  2024-09-04 16:00:00: -0.0126  ['MATICUSDT']
  2024-09-07 08:00:00: -0.0097  ['CRVUSDT', 'MATICUSDT']
  2024-09-06 00:00:00: -0.0058  ['CRVUSDT', 'MATICUSDT']


The worst bars trace to a real perp-market liquidity collapse (verified: volume=0 for several
consecutive days on `ICPUSDT` and `MATICUSDT` — the latter matching 013's own documented MATIC→POL
rebrand feed gap), which the 30-day trailing-median liquidity screen is too slow to catch. Not fixed
here (would be after-the-fact tuning of a frozen parameter, sec 12) — reported as a capacity finding.

## Phase 5 — Controls and ablations (6 exhibits, `n_trials` 12→18 exactly as pre-declared)

In [7]:
phase5 = load("phase_5_18_results.json")
b = phase5["baseline"]
print(f"baseline: net Sharpe {b['sharpe_net']:.3f}")
print()
nh = phase5["1_no_hysteresis"]
print(
    f"1. no-hysteresis: net Sharpe {nh['sharpe_net']:.3f} (turnover {nh['annualized_turnover']:.1f}/yr "
    f"vs baseline {b['annualized_turnover']:.1f}/yr)"
)
print()
po = phase5["2_perp_leg_only_no_spot_hedge"]
print(
    f"2. perp-leg-only: beta_basket={po['beta_to_crypto_basket']:.3f}, beta_btc={po['beta_to_btc']:.3f} "
    f"(hedged: {po['hedged_baseline_beta_to_crypto_basket']:.4f} / {po['hedged_baseline_beta_to_btc']:.4f}), "
    f"max_drawdown={po['max_drawdown']:.1%}"
)
print()
ex = phase5["3_excluding_luna_ftt"]
print(
    f"3. excluding LUNA/FTT: net Sharpe {ex['sharpe_net']:.3f} (baseline {b['sharpe_net']:.3f})"
)
print()
print("4. cost sensitivity (round-turn bp -> gross Sharpe):")
for k, v in phase5["4_cost_sensitivity"].items():
    print(f"   {k}bp: {v['sharpe']:.3f}")
print()
lev = phase5["5_levered_variants"]
liq = phase5["5_liquidation_analysis"]
print(
    f"5. levered: 3x Sharpe={lev['3']['sharpe']:.3f}, 5x Sharpe={lev['5']['sharpe']:.3f}"
)
print(
    f"   1% ES on perp leg's own 8h return: {liq['es_1pct_per_period']:.1%} "
    f"(3x deployed-capital ES: {liq['leveraged_3x_es_1pct_of_deployed_capital']:.1%}, "
    f"5x: {liq['leveraged_5x_es_1pct_of_deployed_capital']:.1%})"
)
print()
print("6. by-year net Sharpe:")
for yr, m in phase5["6_by_year_decomposition"].items():
    print(f"   {yr}: {m['sharpe']:.3f}")

baseline: net Sharpe 0.577

1. no-hysteresis: net Sharpe -0.726 (turnover 95.4/yr vs baseline 56.9/yr)

2. perp-leg-only: beta_basket=-0.897, beta_btc=-1.103 (hedged: 0.0005 / 0.0016), max_drawdown=-99.6%

3. excluding LUNA/FTT: net Sharpe 0.562 (baseline 0.577)

4. cost sensitivity (round-turn bp -> gross Sharpe):
   0.0bp: 2.661
   17.0bp: 1.620
   34.0bp: 0.577
   51.0bp: -0.450

5. levered: 3x Sharpe=0.518, 5x Sharpe=0.455
   1% ES on perp leg's own 8h return: 13.9% (3x deployed-capital ES: 41.8%, 5x: 69.6%)

6. by-year net Sharpe:
   2021: 7.561
   2022: -0.192
   2023: 1.717
   2024: 0.674


## Phase 6 — Holdout: not spent

Gate FA-2 and FA-3 did not both fire on development, so the holdout was never touched. Verified, not
just asserted: `run_phase_6_18_holdout.py` was invoked directly and refused (exit code 1) without
reading `src/research/cache/basis18/holdout/`.

In [8]:
import subprocess
from pathlib import Path

repo_root = Path.cwd().resolve().parents[1]
result = subprocess.run(
    [sys.executable, "src/research/tmp/run_phase_6_18_holdout.py"],
    capture_output=True,
    text=True,
    cwd=str(repo_root),
    check=False,
)
print("exit code:", result.returncode)
print(result.stdout)
print(result.stderr[-2000:] if result.returncode not in (0, 1) else "")

exit code: 1
Phase 4 holdout_access block: {'rule': 'requires FA-2 AND FA-3', 'fa2_fires': False, 'fa3_fires': False, 'access_granted': False}
REFUSED: Gate FA-2 and Gate FA-3 did not both fire on development. Per NEXT_PROMPT sec 8/9.3, this script will not read src/research/cache/basis18/holdout under this condition.




## Bottom line

**Gate FA-1 fires — a genuine, statistically significant funding carry exists in this repo's own
data, driven by funding, not basis drift.** But **Gates FA-2 and FA-3 do not fire**: net Sharpe
(+0.577) clears the absolute bar at every origin offset, and timing shows a large point-estimate edge
over always-on (12x the gross Sharpe), but neither the bootstrap CI on net return nor the paired CI on
timing's own value-add can yet rule out zero, and the deflated-Sharpe estimator — already flagged in
this programme's own methodology notes as likely too harsh for a near-identical-offset trial family —
fails decisively (0.186 vs. the 0.95 bar). **Gate FA-4 fires cleanly and is confirmed two independent
ways**: this is a genuinely delta-neutral book, not a disguised long. This is the first gate in this
programme's thirty-one-gate history where a structurally different mechanism (a cash flow, not a
forecast) is statistically significant before costs and passes its own validity check, without yet
clearing the higher bar of a demonstrably tradeable, holdout-worthy edge. Full detail, including the
concentration/liquidity-screen capacity finding and the two in-flight bugs found and fixed, is in
`src/results/018_funding_basis_trade.md`.